# MedXIR hybrid retrieval: query variants + dense + PRF + BM25 + rerank

Generated by `scripts/build_hybrid_notebook.py`. Edit the generator, not this file.

RTX Pro 6000, Internet OFF. Inputs (all found by glob under `/kaggle/input`):
kernel outputs `medxir-package-models`, `medxir-embed-corpus-4b-s0/s1/s2`,
`medxir-bm25-vi`, `medxir-package-rerankers`; datasets `medxir-retrieval-inputs`,
`ntklinhfitus/medxir-chunk-corpus-v2`, `ntklinhfitus/medxir-vectors-qwen3-4b-additions`.
Attach one full chunk corpus and one copy of each vector part. Share the private ntklinhfitus
datasets with the account running this notebook before starting it.

Steps: input/ID preflight -> query variants -> encode (Qwen3-Embedding-4B) -> exact search over the 2.89M chunk vectors held on
the GPU -> PRF search -> BM25 candidates -> weighted RRF pool -> rerank -> document scores (max/top3) ->
submission ZIP (kept under 100 MB) + `pool.parquet` to rebuild variants offline without a GPU ->
extra ZIPs `medxir-expand-nb60-d200[-drr-bge|-drr-bge-only]-submission.zip`: neighbour spans of the top
EXPAND_NB chunks, documents optionally re-ordered by bge-reranker-v2-m3 (DOC_RERANK, +~25 min for 480k pairs).

Time with every switch on (bge reranker, POOL_MAX 600): about 45-75 min. Rerank ~720k pairs at ~530 pairs/s
(benchmark 26/10) is 20-30 min; the 4B model load (4-9 min) and the 14.8 GB vector read are most of the rest.
Without the reranker: about 20-45 min. Progress and the rerank ETA are written to progress.txt.
BM25 candidates: rerun `kaggle_bm25_vi` on corpus v2 (its bm25_manifest.json states the corpus, checked here); with
the v1 run, chunks added in v2 enter the pool through dense search only.

In [ ]:
# ===================== Config: the only cell to edit =====================
USE_QUERY_SPLIT = True      # rule-based cleanup + split of long queries; each sub-query is searched on its own
USE_CUSTOM_INSTRUCT = True  # task-specific Qwen3-Embedding instruction instead of the model-card default
USE_PRF = True              # pseudo-relevance feedback: query moved toward its top PRF_K chunks, searched again
USE_BM25 = True             # add BM25 (Vietnamese chunks) candidates to the pool
USE_RERANK = True           # cross-encoder scores the fused pool; chunk and document scores come from it
DOC_AGG = 'top3'            # document score: 'max' (old: best chunk) or 'top3' (sum of the 3 best chunks)
LANGS = 'vi,en,zh'          # corpus languages searched, e.g. 'vi' or 'vi,zh' (BM25 covers 'vi' only)
EXPECTED_CORPUS_VERSION = 'v2'  # rejects accidentally attaching the older full corpus
BM25_CORPUS_VERSION = 'v1'  # BM25 run without bm25_manifest.json (the v1 run); newer runs state their corpus

RERANKER = 'bge-reranker-v2-m3'  # 'Qwen3-Reranker-0.6B' is ~3.4x slower, 'Qwen3-Reranker-4B' ~14x (too slow here)
N_DOCS = 200                # 200 or 300: d300 raised Doc R 0.473 -> 0.519 but Doc F2 fell (FINAL 0.2235 -> 0.2125)
N_CHUNKS = 150              # lowered by 10 until the ZIP fits MAX_ZIP_MB (dashboard limit 100 MB)
MAX_ZIP_MB = 99.0

# Extra ZIPs from the same pool after the hybrid ZIP (scripts/expand_chunks.py), named medxir-expand-nb60-d200-*.
EXPAND_NB = 60              # neighbours (order +-1) of the top EXPAND_NB chunks merged into spans (nb60 0.2235); 0 = off
DOC_RERANK = 'fusion'       # bge-reranker-v2-m3 on candidate documents: 'off', 'fusion' (RRF with the chunk-based
                            # order, 0.2253), 'only' (bge order, 0.2253) or 'both' (two ZIPs, scored once)
DOC_RERANK_DEPTH = N_DOCS + 200  # candidate documents scored per query (the cut is at N_DOCS); 29/10 run: 400
DOC_LEAD_CHARS = 400        # document start (title / lead) shown before its best pool chunk

DENSE_K = 500               # candidates of the full query
SUB_K = 100                 # candidates of each sub-query
PRF_K, PRF_ALPHA, PRF_KEEP = 5, 0.3, 300  # feedback chunks, weight of their mean, candidates of the PRF query
BM25_K = 300                # BM25 candidates per query (the attached run kept 500)
POOL_MAX = 600              # fused candidates sent to the reranker per query (pairs = 1,200 x POOL_MAX)
RRF_K = 60
W_SUB, W_PRF, W_BM25 = 0.5, 0.5, 0.3  # RRF weights; the full-query dense list has weight 1
SPLIT_MIN_WORDS, MIN_SUB_WORDS, MAX_SUBQ = 20, 4, 4

INSTRUCT_DEFAULT = 'Given a web search query, retrieve relevant passages that answer the query'
INSTRUCT_CUSTOM = ('Given a Vietnamese patient question about a health or medical problem, retrieve medical '
                   'passages in Vietnamese, English or Chinese that answer the question')
SEARCH_TF32 = True          # TF32 matmul (fp16 vectors are exact in TF32); False = full float32 like the baseline
ENCODE_HERE = True          # False: read query_variant_vectors.npy + query_variants.json from an attached earlier run
QUERY_LIMIT = 0             # >0: first N queries only (speed check; no submission is written)

In [ ]:
import faulthandler
import json
import os
import resource
import time
import traceback
from contextlib import contextmanager
from pathlib import Path

OUT = Path('/kaggle/working')
OUT.mkdir(parents=True, exist_ok=True)
T0 = time.perf_counter()
PROGRESS = OUT / 'progress.txt'
FAULT_FILE = open(OUT / 'faulthandler.txt', 'w')  # native crashes dump the Python stack here
faulthandler.enable(file=FAULT_FILE, all_threads=True)
FAILED = None


def note(message):
    rss_gib = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2 ** 20  # Linux reports KiB
    line = f'{time.perf_counter() - T0:8.1f}s  peak_rss={rss_gib:.1f}GiB  {message}'
    print(line, flush=True)
    with open(PROGRESS, 'a', encoding='utf-8') as file:
        file.write(line + '\n')


@contextmanager
def stage(name):
    # A Python exception is written to retrieval_error.txt and does not stop the kernel, so the
    # session ends COMPLETE and the files can be fetched; later steps are skipped.
    global FAILED
    note(f'start {name}')
    try:
        yield
        note(f'done {name}')
    except Exception:
        FAILED = name
        (OUT / 'retrieval_error.txt').write_text(f'stage: {name}\n\n{traceback.format_exc()}', encoding='utf-8')
        note(f'FAILED {name}; traceback in retrieval_error.txt')

In [ ]:
# Copied from scripts/hybrid_retrieval.py by scripts/build_hybrid_notebook.py. Edit the script and rebuild.
"""CPU-side logic of the hybrid retrieval notebook (scripts/build_hybrid_notebook.py copies this file into it).

Query variants: rule-based cleanup of patient chit-chat and a split of long queries into sentences and
comma clauses; no LLM. Every variant is searched on its own and the lists are fused with weighted RRF.
Document ranking: a document scores the max, or the sum of its 3 best chunk scores (DOC_AGG). Scores
only depend on the submitted sets, so the ranking matters through what falls inside N_DOCS / N_CHUNKS.

With one dense list, DOC_AGG='max' and no reranker, assemble() gives exactly the old d200-c150 lists
(scripts/retrieve_local_4b.py): docs in order of their best chunk, chunk docs appended when missing.
"""
import re
import unicodedata

CHITCHAT = re.compile(
    r'(?i)\b(?:'
    r'(?:xin )?chào (?:bác s[ĩỹi]|bs)[^,.?!]*[,.!]?'
    r'|(?:cho|xin cho) (?:em|tôi|mình|cháu|con|chị|anh) (?:được )?hỏi(?: (?:bác s[ĩỹi]|bs))?(?: là)?'
    r'|(?:em |tôi |mình )?xin hỏi'
    r'|bác s[ĩỹi] ơi'
    r'|(?:em |tôi |mình |xin )?(?:chân thành )?cảm ơn(?: (?:bác s[ĩỹi]|bs))?(?: nhiều)?(?: ạ)?'
    r'|mong (?:bác s[ĩỹi]|bs)[^.?!]*(?:tư vấn|giải đáp|trả lời|giúp)[^.?!]*'
    r'|nhờ (?:bác s[ĩỹi]|bs)[^.?!]*(?:tư vấn|giải đáp)[^.?!]*'
    r')')
SENTENCE = re.compile(r'(?<=[.?!;…])\s+|\n+')


def clean_query(text):
    """Drop greetings, thanks and 'cho em hỏi'-style openers; keep every medical word."""
    text = CHITCHAT.sub(' ', unicodedata.normalize('NFC', text))
    return re.sub(r'\s+', ' ', text).strip(' ,.')


def sub_queries(query, split_min_words=20, min_sub_words=4, max_subq=4):
    """Sentences of a long query, plus the comma clauses of a sentence when every clause has 3+ words.

    Short queries (most are one sentence of ~18 words) return []: they are searched whole only.
    """
    if len(query.split()) < split_min_words:
        return []
    found = []
    for sentence in SENTENCE.split(clean_query(query)):
        sentence = sentence.strip(' ,.;')
        if len(sentence.split()) < min_sub_words:
            continue
        found.append(sentence)
        clauses = [part.strip() for part in sentence.split(',')]
        if len(clauses) >= 2 and all(len(part.split()) >= 3 for part in clauses):
            found.extend(clauses)
    seen, out = {query.lower().strip()}, []
    for text in found:
        if text.lower() not in seen:
            seen.add(text.lower())
            out.append(text)
    return out[:max_subq]


def rrf(lists, k=60):
    """Weighted reciprocal-rank fusion. lists: [(ranked ids, weight)]. Returns [(id, score)], best first.

    Ties keep the order of first appearance, so a single list comes back in its own order.
    """
    score, first = {}, {}
    for ids, weight in lists:
        for rank, cid in enumerate(ids):
            score[cid] = score.get(cid, 0.0) + weight / (k + rank + 1)
            first.setdefault(cid, len(first))
    return sorted(score.items(), key=lambda item: (-item[1], first[item[0]]))


def doc_ranking(scored, docs_of, agg='max'):
    """Documents of the scored chunks ([(chunk id, score)], best first), best document first.

    agg='max': a document scores its best chunk (the old ranking). agg='top3': the sum of its 3 best chunks,
    so a document with several matching chunks beats one with a single lucky match.
    """
    per_doc, first = {}, {}
    for cid, score in scored:
        for doc_id in docs_of(cid):
            per_doc.setdefault(doc_id, []).append(score)
            first.setdefault(doc_id, len(first))
    if agg == 'max':
        value = {doc: max(scores) for doc, scores in per_doc.items()}
    elif agg == 'top3':
        value = {doc: sum(sorted(scores, reverse=True)[:3]) for doc, scores in per_doc.items()}
    else:
        raise ValueError(f'unknown DOC_AGG {agg!r}')
    return sorted(per_doc, key=lambda doc: (-value[doc], first[doc]))


def assemble(scored, fill_ids, docs_of, n_docs, n_chunks, agg='max'):
    """One query's (relevant_docs, [(doc_id, chunk id)]).

    scored: [(chunk id, score)] best first, the chunks with a final score (the reranked pool, or all fused
    candidates). fill_ids: further candidate chunk ids in fused order, used only to top the document list
    up to n_docs. Chunk documents missing from the list are appended, as in the earlier submissions.
    """
    docs = doc_ranking(scored, docs_of, agg)[:n_docs]
    listed = set(docs)
    for cid in fill_ids:
        if len(docs) >= n_docs:
            break
        for doc_id in docs_of(cid):
            if doc_id not in listed and len(docs) < n_docs:
                listed.add(doc_id)
                docs.append(doc_id)
    chunks = []
    for cid, _ in scored[:n_chunks]:
        for doc_id in docs_of(cid):
            chunks.append((doc_id, cid))
            if doc_id not in listed:
                listed.add(doc_id)
                docs.append(doc_id)
    return docs, chunks

# Input validation copied from scripts/hybrid_inputs.py.
"""CPU preflight for hybrid retrieval: part discovery, coverage and row alignment."""
import hashlib
import json
from collections import Counter
from pathlib import Path


def unique_parts(paths, chunk=False):
    result = {}
    for raw in paths:
        path = Path(raw)
        key = f'{path.parent.name}-{path.stem}' if chunk else path.stem
        assert key.split('-')[0] in {'vi', 'en', 'zh'}, f'Unknown language in part: {path}'
        assert key not in result, f'Duplicate part {key}: {result.get(key)} and {path}'
        result[key] = str(path)
    assert result, 'No chunk parts found' if chunk else 'No vector parts found'
    return result


def select_input_parts(chunk_files, vector_files, langs):
    langs = list(langs)
    assert langs and len(set(langs)) == len(langs) and set(langs) <= {'vi', 'en', 'zh'}, f'Invalid LANGS: {langs}'
    chunks = unique_parts(chunk_files, chunk=True)
    vectors = unique_parts(vector_files)
    chunks = {k: p for k, p in chunks.items() if k.split('-')[0] in langs}
    vectors = {k: p for k, p in vectors.items() if k.split('-')[0] in langs}
    assert {k.split('-')[0] for k in chunks} == set(langs), f'Missing chunk languages: {langs}'
    assert set(chunks) == set(vectors), (
        f'Chunk/vector parts differ: missing vectors={sorted(set(chunks)-set(vectors))}; '
        f'extra vectors={sorted(set(vectors)-set(chunks))}')
    return chunks, vectors


def json_fingerprint(path):
    path = Path(path)
    return dict(path=str(path), sha256=hashlib.sha256(path.read_bytes()).hexdigest())


def validate_hybrid_inputs(chunk_files, vector_files, langs, manifest_path, expected_version='v2', progress=None):
    import numpy as np
    import pyarrow.parquet as pq

    manifest_path = Path(manifest_path)
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    assert not expected_version or manifest['dataset_version'] == expected_version, (
        f'Expected corpus {expected_version}, found {manifest.get("dataset_version")}')
    chunks, vectors = select_input_parts(chunk_files, vector_files, langs)
    schemas = {}; rows = Counter(); seen = set(); vector_manifests = {}
    for key in sorted(vectors):
        path = Path(vectors[key]); chunk_path = Path(chunks[key]); lang = key.split('-')[0]
        assert chunk_path.resolve().is_relative_to(manifest_path.parent.resolve()), f'Chunk part outside corpus root: {chunk_path}'
        embed_manifest_path = path.parent.parent / 'embed_manifest.json'
        if str(embed_manifest_path) not in vector_manifests:
            m = json.loads(embed_manifest_path.read_text(encoding='utf-8'))
            assert m['complete'] and m['model'] == 'Qwen/Qwen3-Embedding-4B' and m['dim'] == 2560, f'Incompatible vector manifest: {embed_manifest_path}'
            assert m['max_length'] == 1024 and m['dtype'] == 'torch.bfloat16', f'Incompatible embedding contract: {embed_manifest_path}'
            for field, expected in [('instruction', ''), ('pooling', 'last_token'), ('padding_side', 'left'), ('normalize', True)]:
                assert m.get(field, expected) == expected, f'Incompatible {field}: {embed_manifest_path}'
            vector_manifests[str(embed_manifest_path)] = dict(**json_fingerprint(embed_manifest_path), model=m['model'], dim=m['dim'],
                model_revision=m.get('model_revision'), output_dtype=m.get('output_dtype', 'float16'))
        array = np.load(path, mmap_mode='r', allow_pickle=False)
        assert array.ndim == 2 and array.dtype == np.float16 and array.shape[1] == 2560, (key, array.dtype, array.shape)
        ids_path = path.with_name(f'{key}-ids.parquet')
        ids_table = pq.read_table(ids_path, columns=['chunk_id'])
        chunk_table = pq.read_table(chunk_path, columns=['chunk_id', 'lang'])
        assert ids_table['chunk_id'].null_count == chunk_table['chunk_id'].null_count == 0, f'Null ID in {key}'
        ids = ids_table['chunk_id'].to_numpy(); expected_ids = chunk_table['chunk_id'].to_numpy()
        assert array.shape[0] == len(ids) == len(expected_ids) and len(ids) > 0, f'Row count mismatch: {key}'
        assert np.array_equal(ids, expected_ids), f'Chunk ID order mismatch: {key}'
        assert set(chunk_table['lang'].to_pylist()) == {lang}, f'Wrong language rows: {key}'
        part_ids = set(ids.tolist())
        assert len(part_ids) == len(ids) and not seen.intersection(part_ids), f'Duplicate chunk IDs: {key}'
        seen.update(part_ids); rows[lang] += len(ids)
        schemas[key] = dict(chunk_path=str(chunk_path), vector_path=str(path), ids_path=str(ids_path), rows=len(ids), language=lang)
        del array, ids, expected_ids, ids_table, chunk_table, part_ids
        if progress: progress(f'preflight {key}: {schemas[key]["rows"]:,} aligned IDs')
    for lang in langs:
        assert rows[lang] == manifest['languages'][lang]['chunks'], f'Corpus manifest count mismatch for {lang}: {rows[lang]}'
    provenance = dict(corpus_version=manifest['dataset_version'], corpus_manifest=json_fingerprint(manifest_path),
        selected_languages=list(langs), chunk_count=sum(rows.values()), vector_count=sum(rows.values()),
        rows_by_language=dict(rows), part_count=len(schemas), parts=schemas,
        vector_manifests=list(vector_manifests.values()), all_part_ids_aligned=True)
    return chunks, vectors, provenance

## 1. Inputs and GPU check

In [ ]:
if FAILED is None:
    with stage('setup'):
        import glob
        import subprocess
        import zipfile

        import numpy as np
        import pandas as pd
        import pyarrow as pa
        import pyarrow.compute as pc
        import pyarrow.parquet as pq
        import torch

        CONFIG = {name: value for name, value in globals().items()
                  if name.isupper() and name not in {'OUT', 'T0', 'PROGRESS', 'FAULT_FILE', 'FAILED'}
                  and isinstance(value, (bool, int, float, str))}
        TIMINGS = {}


        def one_match(pattern, description):
            hits = sorted(glob.glob(pattern, recursive=True))
            assert len(hits) == 1, f'{description}: expected one match, found {hits[:10]}'
            return Path(hits[0])


        # The attached inputs, so a missing or doubled source shows in the log before any check fails.
        for root in sorted(glob.glob('/kaggle/input/*/*/*')) or sorted(glob.glob('/kaggle/input/*')):
            print('input:', root, sorted(os.listdir(root))[:12] if os.path.isdir(root) else '')

        QUERY_PATH = one_match('/kaggle/input/**/query.parquet', 'query.parquet')
        CORPUS_PATH = one_match('/kaggle/input/**/links_corpus_vi_en_zh.parquet', 'official corpus ids')
        HASH_PATH = one_match('/kaggle/input/**/hash_groups.parquet', 'hash_groups.parquet (medxir-chunk-corpus)')
        CHUNK_MANIFEST_PATH = HASH_PATH.parent / 'chunk_manifest.json'
        assert CHUNK_MANIFEST_PATH.is_file(), f'Missing corpus manifest: {CHUNK_MANIFEST_PATH}'
        CHUNK_FILES = sorted(glob.glob('/kaggle/input/**/chunks/*/part-*.parquet', recursive=True))
        VECTOR_FILES = sorted(p for p in glob.glob('/kaggle/input/**/vectors/*.npy', recursive=True) if not p.endswith('.tmp.npy'))
        SEARCH_LANGS = [lang.strip() for lang in LANGS.split(',') if lang.strip()]
        CHUNK_PATH, VECTOR_PATH, INPUT_PROVENANCE = validate_hybrid_inputs(
            CHUNK_FILES, VECTOR_FILES, SEARCH_LANGS, CHUNK_MANIFEST_PATH, EXPECTED_CORPUS_VERSION, progress=note)
        CHUNK_FILES = [CHUNK_PATH[key] for key in sorted(CHUNK_PATH)]
        note(f'inputs verified: {INPUT_PROVENANCE["part_count"]} parts, {INPUT_PROVENANCE["vector_count"]:,} vectors, '
             f'{INPUT_PROVENANCE["vector_count"] * 2560 * 2 / 1e9:.2f} GB fp16; languages={SEARCH_LANGS}')
        BM25_ACTIVE = USE_BM25 and 'vi' in SEARCH_LANGS
        BM25_PROVENANCE = dict(requested=USE_BM25, active=BM25_ACTIVE, corpus_version=BM25_CORPUS_VERSION,
                               coverage='not_used', candidates_path=None)
        if ENCODE_HERE:
            MODEL_PATH = one_match('/kaggle/input/**/Qwen3-Embedding-4B/config.json', 'packaged Qwen3-Embedding-4B').parent
        if BM25_ACTIVE:
            BM25_PATH = one_match('/kaggle/input/**/vi-candidates-k500.npz', 'BM25 candidates (medxir-bm25-vi)')
            bm25_manifest = BM25_PATH.with_name('bm25_manifest.json')
            if bm25_manifest.is_file():   # written by the BM25 runs since corpus v2; older runs only have the config label
                bm25_info = json.loads(bm25_manifest.read_text(encoding='utf-8'))
                BM25_CORPUS_VERSION = bm25_info['corpus_version']
                if BM25_CORPUS_VERSION == INPUT_PROVENANCE['corpus_version'] and 'vi' in SEARCH_LANGS:
                    assert bm25_info['vi_chunks'] == INPUT_PROVENANCE['rows_by_language']['vi'], ('BM25 / corpus VI chunks differ', bm25_info)
            BM25_PROVENANCE.update(corpus_version=BM25_CORPUS_VERSION,candidates_path=str(BM25_PATH), coverage='same_corpus' if BM25_CORPUS_VERSION == INPUT_PROVENANCE['corpus_version'] else 'older_or_different_corpus')
            if BM25_PROVENANCE['coverage'] != 'same_corpus':
                note(f'BM25 candidates cover {BM25_CORPUS_VERSION}; current corpus is {INPUT_PROVENANCE["corpus_version"]}. '
                     'Added chunks enter through dense search until BM25 is rebuilt.')
        if USE_RERANK:
            RERANK_PATH = one_match(f'/kaggle/input/**/{RERANKER}/config.json', f'packaged {RERANKER}').parent
        assert DOC_RERANK in {'off', 'fusion', 'only', 'both'}, f'unknown DOC_RERANK {DOC_RERANK!r}'
        assert DOC_RERANK == 'off' or DOC_RERANK_DEPTH > N_DOCS, 'DOC_RERANK_DEPTH must reach past the N_DOCS cut'
        if DOC_RERANK != 'off':
            DOC_RERANK_PATH = one_match('/kaggle/input/**/bge-reranker-v2-m3/config.json', 'packaged bge-reranker-v2-m3').parent

        queries = pd.read_parquet(QUERY_PATH, columns=['id', 'query'])
        assert len(queries) == 1200 and queries['id'].is_unique
        if QUERY_LIMIT:
            queries = queries.head(QUERY_LIMIT).reset_index(drop=True)
        QUERY_IDS = [int(q) for q in queries['id']]
        QUERY_TEXT = queries['query'].astype(str).tolist()

        print(subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total,driver_version', '--format=csv'],
                             capture_output=True, text=True).stdout)
        GPU_NAME = torch.cuda.get_device_name(0)
        assert '6000' in GPU_NAME, f'Expected RTX Pro 6000, got {GPU_NAME!r}'
        print('GPU:', GPU_NAME, '| queries:', len(queries), '| vector parts:', list(VECTOR_PATH))
        print(json.dumps(CONFIG, ensure_ascii=False, indent=1))

## 2. Query variants (USE_QUERY_SPLIT)

In [ ]:
if FAILED is None:
    with stage('variants'):
        # One row per text to encode: the full query (always) and its sub-queries (USE_QUERY_SPLIT).
        VARIANTS = []
        for qi, text in enumerate(QUERY_TEXT):
            VARIANTS.append(dict(qi=qi, kind='full', text=text))
            if USE_QUERY_SPLIT:
                for sub in sub_queries(text, SPLIT_MIN_WORDS, MIN_SUB_WORDS, MAX_SUBQ):
                    VARIANTS.append(dict(qi=qi, kind='sub', text=sub))
        TASK = INSTRUCT_CUSTOM if USE_CUSTOM_INSTRUCT else INSTRUCT_DEFAULT
        (OUT / 'query_variants.json').write_text(json.dumps(dict(task=TASK, variants=VARIANTS), ensure_ascii=False) + '\n',
                                                 encoding='utf-8')
        n_sub = sum(v['kind'] == 'sub' for v in VARIANTS)
        print(f'{len(VARIANTS)} texts to encode: {len(QUERY_TEXT)} full, {n_sub} sub-queries '
              f'({len({v["qi"] for v in VARIANTS if v["kind"] == "sub"})} queries split)')

## 3. Encode with Qwen3-Embedding-4B (USE_CUSTOM_INSTRUCT picks the instruction)

In [ ]:
if FAILED is None:
    with stage('encode'):
        started = time.perf_counter()
        if ENCODE_HERE:
            from transformers import AutoModel, AutoTokenizer
            import torch.nn.functional as F

            os.environ['HF_HUB_OFFLINE'] = '1'
            tok = AutoTokenizer.from_pretrained(str(MODEL_PATH))
            tok.padding_side = 'left'
            model = AutoModel.from_pretrained(str(MODEL_PATH), torch_dtype=torch.bfloat16, attn_implementation='sdpa').cuda().eval()
            note(f'embedding model loaded in {time.perf_counter() - started:.0f}s')

            def last_token_pool(hidden, mask):
                if mask[:, -1].sum() == mask.shape[0]:
                    return hidden[:, -1]
                lengths = mask.sum(dim=1) - 1
                return hidden[torch.arange(hidden.shape[0], device=hidden.device), lengths]

            @torch.inference_mode()
            def encode(texts, batch=16):
                tokenized = tok(texts, truncation=True, max_length=1024)['input_ids']
                order = np.argsort([-len(row) for row in tokenized], kind='stable')
                vectors = np.empty((len(texts), model.config.hidden_size), dtype=np.float32)
                for start in range(0, len(order), batch):
                    index = order[start:start + batch]
                    enc = tok.pad({'input_ids': [tokenized[i] for i in index]}, return_tensors='pt').to('cuda')
                    hidden = model(input_ids=enc['input_ids'], attention_mask=enc['attention_mask'], use_cache=False).last_hidden_state
                    vectors[index] = F.normalize(last_token_pool(hidden, enc['attention_mask']).float(), p=2, dim=1).cpu().numpy()
                return vectors

            VARIANT_VECTORS = encode([f'Instruct: {TASK}\nQuery:{v["text"]}' for v in VARIANTS])
            del model, tok
            torch.cuda.empty_cache()
            np.save(OUT / 'query_variant_vectors.npy', VARIANT_VECTORS)
        else:
            saved = json.loads(one_match('/kaggle/input/**/query_variants.json', 'earlier query_variants.json').read_text(encoding='utf-8'))
            assert saved['variants'] == VARIANTS and saved['task'] == TASK, 'attached variants differ from this config'
            VARIANT_VECTORS = np.load(one_match('/kaggle/input/**/query_variant_vectors.npy', 'earlier variant vectors'))
        assert VARIANT_VECTORS.shape == (len(VARIANTS), 2560) and np.isfinite(VARIANT_VECTORS).all()
        TIMINGS['encode_s'] = round(time.perf_counter() - started, 1)

## 4. Load the selected vector parts onto the GPU

In [ ]:
if FAILED is None:
    with stage('load_vectors'):
        # Selected parts stay on the GPU as float16 (all v2: 2.89M x 2560 = 14.8 GB), so the full, sub and PRF
        # searches read /kaggle/input once. Blocks of 32k rows through mmap: no 1-3 GB single read.
        READ_ROWS = 32768
        started = time.perf_counter()
        PART_KEYS = sorted(VECTOR_PATH)
        PARTS, IDS = [], []
        for key in PART_KEYS:
            t0 = time.perf_counter()
            array = np.load(VECTOR_PATH[key], mmap_mode='r', allow_pickle=False)
            assert array.dtype == np.float16 and array.shape[1] == 2560, (key, array.dtype, array.shape)
            gpu = torch.empty(array.shape, dtype=torch.float16, device='cuda')
            for start in range(0, array.shape[0], READ_ROWS):
                gpu[start:start + READ_ROWS] = torch.from_numpy(np.array(array[start:start + READ_ROWS])).cuda()
            del array
            ids_path = Path(VECTOR_PATH[key]).with_name(f'{key}-ids.parquet')
            ids = pq.read_table(ids_path, columns=['chunk_id']).column('chunk_id').to_numpy()
            assert len(ids) == gpu.shape[0], key
            PARTS.append(gpu)
            IDS.append(ids)
            seconds = time.perf_counter() - t0
            note(f'{key}: {gpu.shape[0]:,} rows on GPU in {seconds:.0f}s ({gpu.numel() * 2 / 1e6 / max(seconds, 1e-9):.0f} MB/s)')
        OFFSETS = np.cumsum([0] + [p.shape[0] for p in PARTS])
        ALL_IDS = np.concatenate(IDS)
        del IDS
        assert len(set(ALL_IDS.tolist())) == len(ALL_IDS), 'chunk_id repeated across vector parts'
        TIMINGS['load_vectors_s'] = round(time.perf_counter() - started, 1)

## 5. Exact dense search: full query, sub-queries, PRF (USE_PRF)

In [ ]:
if FAILED is None:
    with stage('search'):
        torch.backends.cuda.matmul.allow_tf32 = SEARCH_TF32
        ROW_BLOCK, QUERY_BATCH = 262144, 1024


        @torch.inference_mode()
        def search(query_vectors, k):
            # Exact inner product over every part; returns (scores, global row) shaped (queries, k), best first.
            q = torch.from_numpy(np.ascontiguousarray(query_vectors, dtype=np.float32)).cuda()
            best_s = torch.full((len(q), k), -float('inf'), device='cuda')
            best_i = torch.full((len(q), k), -1, dtype=torch.int64, device='cuda')
            for part, offset in zip(PARTS, OFFSETS[:-1]):
                for row in range(0, part.shape[0], ROW_BLOCK):
                    block = part[row:row + ROW_BLOCK].float()
                    for qs in range(0, len(q), QUERY_BATCH):
                        scores = q[qs:qs + QUERY_BATCH] @ block.T
                        top_s, top_i = torch.topk(scores, min(k, scores.shape[1]), dim=1)
                        cat_s = torch.cat([best_s[qs:qs + QUERY_BATCH], top_s], dim=1)
                        cat_i = torch.cat([best_i[qs:qs + QUERY_BATCH], top_i + int(offset) + row], dim=1)
                        keep_s, order = torch.topk(cat_s, k, dim=1)
                        best_s[qs:qs + QUERY_BATCH] = keep_s
                        best_i[qs:qs + QUERY_BATCH] = torch.gather(cat_i, 1, order)
                    del block
            return best_s.cpu().numpy(), best_i.cpu().numpy()


        def gather_vectors(rows):
            # float32 vectors of global rows (any shape), read from the GPU-resident parts.
            flat = np.asarray(rows).ravel()
            part_of = np.searchsorted(OFFSETS, flat, side='right') - 1
            out = torch.empty((len(flat), 2560), dtype=torch.float32, device='cuda')
            for p in np.unique(part_of):
                mask = np.flatnonzero(part_of == p)
                local = torch.from_numpy(flat[mask] - OFFSETS[p]).cuda()
                out[torch.from_numpy(mask).cuda()] = PARTS[p].index_select(0, local).float()
            return out.view(*np.shape(rows), 2560)


        started = time.perf_counter()
        kinds = np.array([v['kind'] for v in VARIANTS])
        owner = np.array([v['qi'] for v in VARIANTS])
        d_scores, d_rows = search(VARIANT_VECTORS, DENSE_K)
        note(f'dense search: {len(VARIANTS)} vectors in {time.perf_counter() - started:.0f}s')
        FULL = np.flatnonzero(kinds == 'full')
        assert (owner[FULL] == np.arange(len(QUERY_TEXT))).all()
        DENSE_FULL_IDS = ALL_IDS[d_rows[FULL]]               # (queries, DENSE_K) chunk ids, best first
        DENSE_FULL_SCORES = d_scores[FULL]
        SUB_LISTS = [[] for _ in QUERY_TEXT]
        for i in np.flatnonzero(kinds == 'sub'):
            SUB_LISTS[owner[i]].append(ALL_IDS[d_rows[i, :SUB_K]])

        PRF_IDS = None
        if USE_PRF:
            t0 = time.perf_counter()
            with torch.inference_mode():
                feedback = gather_vectors(d_rows[FULL, :PRF_K]).mean(dim=1)
                q = torch.from_numpy(VARIANT_VECTORS[FULL]).cuda()
                prf = torch.nn.functional.normalize((1 - PRF_ALPHA) * q + PRF_ALPHA * feedback, p=2, dim=1).cpu().numpy()
            _, prf_rows = search(prf, PRF_KEEP)
            PRF_IDS = ALL_IDS[prf_rows]
            note(f'PRF search in {time.perf_counter() - t0:.0f}s')

        del PARTS
        torch.cuda.empty_cache()
        TIMINGS['search_s'] = round(time.perf_counter() - started, 1)

## 6. BM25 candidates (USE_BM25)

In [ ]:
if FAILED is None:
    with stage('bm25'):
        BM25_IDS = [[] for _ in QUERY_TEXT]
        if BM25_ACTIVE:
            with np.load(BM25_PATH, allow_pickle=True) as z:  # each z[key] access re-reads the array: read once
                bm25_qids, bm25_scores, bm25_ids = z['query_ids'], z['scores'], z['chunk_ids']
            row_of = {int(q): i for i, q in enumerate(bm25_qids)}
            for qi, qid in enumerate(QUERY_IDS):
                r = row_of[qid]
                keep = bm25_scores[r, :BM25_K] > 0  # no match = score 0
                BM25_IDS[qi] = bm25_ids[r, :BM25_K][keep]
            known = set(ALL_IDS.tolist())
            unknown = sum(cid not in known for ids in BM25_IDS for cid in ids)
            assert unknown == 0, f'{unknown} BM25 chunk ids are not in the vector parts (different chunk corpus?)'
            print('BM25 candidates per query: median', int(np.median([len(x) for x in BM25_IDS])))

## 7. Weighted RRF pool

In [ ]:
if FAILED is None:
    with stage('fuse'):
        # Weighted RRF over the candidate lists; the pool is what gets a final score (reranked when USE_RERANK).
        ONLY_DENSE = not (USE_QUERY_SPLIT or USE_PRF or BM25_ACTIVE)
        FUSED, POOLS = [], []
        for qi in range(len(QUERY_TEXT)):
            lists = [(DENSE_FULL_IDS[qi].tolist(), 1.0)]
            lists += [(ids.tolist(), W_SUB) for ids in SUB_LISTS[qi]]
            if PRF_IDS is not None:
                lists.append((PRF_IDS[qi].tolist(), W_PRF))
            if len(BM25_IDS[qi]):
                lists.append((list(BM25_IDS[qi]), W_BM25))
            fused = rrf(lists, RRF_K)
            FUSED.append(fused)
            POOLS.append(fused[:POOL_MAX] if USE_RERANK else fused)
        pool_sizes = [len(p) for p in POOLS]
        print('fused candidates per query: median', int(np.median([len(f) for f in FUSED])),
              '| pool: median', int(np.median(pool_sizes)), 'total', sum(pool_sizes))

## 8. Chunk text and document groups

In [ ]:
if FAILED is None:
    with stage('records'):
        # Chunk records for every fused candidate (documents); text only for the chunks that need it.
        started = time.perf_counter()
        need_text = set()
        for qi, pool in enumerate(POOLS):
            for cid, _ in (pool if USE_RERANK else pool[:N_CHUNKS]):
                need_text.add(cid)
            need_text.update(DENSE_FULL_IDS[qi, :N_CHUNKS].tolist())  # for the overlap check against plain dense
        wanted = {cid for fused in FUSED for cid, _ in fused}
        wanted_arr, text_arr = pa.array(sorted(wanted)), pa.array(sorted(need_text))
        RECORDS = {}
        for path in CHUNK_FILES:
            table = pq.read_table(path, columns=['chunk_id', 'text_sha256', 'rep_doc_id', 'order', 'chunk_text'])
            table = table.filter(pc.is_in(table['chunk_id'], value_set=wanted_arr))
            with_text = pc.is_in(table['chunk_id'], value_set=text_arr).to_pylist()
            for cid, sha, doc, order, text, keep in zip(table['chunk_id'].to_pylist(), table['text_sha256'].to_pylist(),
                                                        table['rep_doc_id'].to_pylist(), table['order'].to_pylist(),
                                                        table['chunk_text'].to_pylist(), with_text):
                RECORDS[cid] = (sha, int(doc), int(order), text if keep else None)
            del table
        missing = wanted - set(RECORDS)
        assert not missing, f'{len(missing)} candidate chunks have no row in the chunk corpus, e.g. {sorted(missing)[:3]}'

        groups = pd.read_parquet(HASH_PATH, columns=['text_sha256', 'doc_ids', 'fanout_safe'])
        GROUP = {sha: [int(d) for d in doc_ids] for sha, doc_ids, safe in
                 zip(groups['text_sha256'], groups['doc_ids'], groups['fanout_safe']) if bool(safe)}
        del groups


        def docs_of(cid):
            sha, doc, _, _ = RECORDS[cid]
            return GROUP.get(sha) or [doc]


        TIMINGS['records_s'] = round(time.perf_counter() - started, 1)
        print(f'{len(RECORDS):,} candidate chunks, {len(need_text):,} with text, {len(GROUP):,} fan-out groups')

## 9. Rerank the pool (USE_RERANK)

In [ ]:
if FAILED is None:
    with stage('rerank'):
        # Final chunk scores. Reranker: P(relevant) of (full query, chunk). Without it: fused RRF score, or the
        # cosine when the full-query dense list is the only list (then DOC_AGG='max' is the old ranking exactly).
        started = time.perf_counter()
        FINAL = []
        if USE_RERANK:
            from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer

            pairs = [(QUERY_TEXT[qi], RECORDS[cid][3]) for qi, pool in enumerate(POOLS) for cid, _ in pool]
            scores = np.empty(len(pairs), dtype=np.float32)
            order = np.argsort([len(d) for _, d in pairs], kind='stable')  # similar lengths per batch: less padding
            if RERANKER.startswith('Qwen3-Reranker'):
                tok = AutoTokenizer.from_pretrained(str(RERANK_PATH), padding_side='left')
                model = AutoModelForCausalLM.from_pretrained(str(RERANK_PATH), torch_dtype=torch.bfloat16).cuda().eval()
                yes, no = tok.convert_tokens_to_ids('yes'), tok.convert_tokens_to_ids('no')
                prefix = ('<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the '
                          'Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n')
                suffix = '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
                instruct = 'Given a Vietnamese medical question, judge whether the document contains the answer to the question'
                batch = 16

                def score_batch(index):
                    texts = [f'{prefix}<Instruct>: {instruct}\n<Query>: {pairs[i][0]}\n<Document>: {pairs[i][1]}{suffix}' for i in index]
                    enc = tok(texts, padding=True, truncation=True, max_length=1024, return_tensors='pt').to('cuda')
                    logits = model(**enc).logits[:, -1, :]
                    two = torch.stack([logits[:, no], logits[:, yes]], dim=1).float()
                    return torch.softmax(two, dim=1)[:, 1]
            else:
                tok = AutoTokenizer.from_pretrained(str(RERANK_PATH))
                model = AutoModelForSequenceClassification.from_pretrained(str(RERANK_PATH), torch_dtype=torch.bfloat16).cuda().eval()
                batch = 32

                def score_batch(index):
                    enc = tok([pairs[i][0] for i in index], [pairs[i][1] for i in index], padding=True, truncation=True,
                              max_length=1024, return_tensors='pt').to('cuda')
                    return torch.sigmoid(model(**enc).logits.view(-1).float())

            note(f'{RERANKER} loaded; {len(pairs):,} pairs')
            t0, next_report = time.perf_counter(), 0.05
            with torch.inference_mode():
                for start in range(0, len(order), batch):
                    index = order[start:start + batch]
                    scores[index] = score_batch(index).cpu().numpy()
                    done = (start + len(index)) / len(order)
                    if done >= next_report:
                        rate = (start + len(index)) / (time.perf_counter() - t0)
                        note(f'rerank {done:.0%} | {rate:.0f} pairs/s | ETA {(len(order) - start) / rate / 60:.0f} min')
                        next_report += 0.05
            del model, tok
            torch.cuda.empty_cache()
            TIMINGS['rerank_pairs_per_s'] = round(len(pairs) / (time.perf_counter() - t0), 1)
            position = 0
            for pool in POOLS:
                part = scores[position:position + len(pool)]
                position += len(pool)
                ranked = sorted(zip([cid for cid, _ in pool], part.tolist()), key=lambda item: -item[1])
                FINAL.append(ranked)
        elif ONLY_DENSE:
            FINAL = [list(zip(DENSE_FULL_IDS[qi].tolist(), DENSE_FULL_SCORES[qi].tolist())) for qi in range(len(QUERY_TEXT))]
        else:
            FINAL = [list(pool) for pool in POOLS]
        TIMINGS['rerank_s'] = round(time.perf_counter() - started, 1)

## 10. Documents (DOC_AGG), chunks, ZIP under the size limit, manifest

In [ ]:
if FAILED is None:
    with stage('submit'):
        started = time.perf_counter()
        corpus_ids = set(map(int, pd.read_parquet(CORPUS_PATH, columns=['id'])['id']))


        def build(n_chunks):
            submission = []
            for qi, qid in enumerate(QUERY_IDS):
                scored_ids = {cid for cid, _ in FINAL[qi]}
                fill = [cid for cid, _ in FUSED[qi] if cid not in scored_ids]
                docs, chunks = assemble(FINAL[qi], fill, docs_of, N_DOCS, n_chunks, DOC_AGG)
                submission.append(dict(id=qid, relevant_docs=docs, relevant_chunks=[
                    dict(doc_id=doc, chunk_text=RECORDS[cid][3], chunk_order=RECORDS[cid][2]) for doc, cid in chunks]))
            for entry in submission:
                listed = set(entry['relevant_docs'])
                assert entry['relevant_chunks'] and listed <= corpus_ids and len(listed) == len(entry['relevant_docs'])
                assert all(c['doc_id'] in listed and c['chunk_text'] and c['chunk_text'].strip() for c in entry['relevant_chunks'])
            return submission


        def write(submission):
            json_path, zip_path = OUT / 'medxir-hybrid-submission.json', OUT / 'medxir-hybrid-submission.zip'
            json_path.write_text(json.dumps(submission, ensure_ascii=False, separators=(',', ':')), encoding='utf-8')
            with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=9) as archive:
                archive.write(json_path, arcname=json_path.name)
            with zipfile.ZipFile(zip_path) as archive:
                assert archive.namelist() == [json_path.name]
            json_path.unlink()
            return zip_path


        # How far the new lists moved from plain dense (same sets = same score; PROGRESS rule before submitting).
        dense_docs = [doc_ranking(list(zip(DENSE_FULL_IDS[qi].tolist(), DENSE_FULL_SCORES[qi].tolist())), docs_of, 'max')[:N_DOCS]
                      for qi in range(len(QUERY_TEXT))]
        n_chunks = N_CHUNKS
        while True:
            submission = build(n_chunks)
            if QUERY_LIMIT:
                print('QUERY_LIMIT set: submission not written')
                zip_path = None
                break
            zip_path = write(submission)
            zip_mb = zip_path.stat().st_size / 1e6
            note(f'{n_chunks} chunks/query -> ZIP {zip_mb:.1f} MB')
            if zip_mb <= MAX_ZIP_MB:
                break
            n_chunks -= 10

        chunk_overlap = np.mean([len({cid for cid, _ in FINAL[qi][:n_chunks]} & set(DENSE_FULL_IDS[qi, :n_chunks].tolist())) / n_chunks
                                 for qi in range(len(QUERY_TEXT))])
        doc_overlap = np.mean([len(set(e['relevant_docs'][:N_DOCS]) & set(dense_docs[qi])) / max(len(dense_docs[qi]), 1)
                               for qi, e in enumerate(submission)])

        pool_rows = []
        for qi, qid in enumerate(QUERY_IDS):
            dense_rank = {cid: r for r, cid in enumerate(DENSE_FULL_IDS[qi].tolist())}
            bm25_rank = {cid: r for r, cid in enumerate(BM25_IDS[qi])}
            fused = dict(FUSED[qi])
            for rank, (cid, score) in enumerate(FINAL[qi]):
                pool_rows.append((qid, cid, rank, float(score), fused.get(cid, 0.0), dense_rank.get(cid, -1), bm25_rank.get(cid, -1)))
        pd.DataFrame(pool_rows, columns=['query_id', 'chunk_id', 'final_rank', 'final_score', 'rrf', 'dense_rank', 'bm25_rank']) \
          .to_parquet(OUT / 'pool.parquet', index=False, compression='zstd')
        TIMINGS['submit_s'] = round(time.perf_counter() - started, 1)

        manifest = dict(
            config=CONFIG, task_instruction=TASK, gpu=GPU_NAME, queries=len(QUERY_IDS),
            inputs=INPUT_PROVENANCE, bm25=BM25_PROVENANCE,
            sub_queries=int(sum(v['kind'] == 'sub' for v in VARIANTS)), chunks_per_query=n_chunks, docs_per_query=N_DOCS,
            zip=str(zip_path) if zip_path else None, zip_mb=round(zip_path.stat().st_size / 1e6, 1) if zip_path else None,
            pool_median=int(np.median([len(p) for p in POOLS])),
            chunk_overlap_with_dense=round(float(chunk_overlap), 4), doc_overlap_with_dense=round(float(doc_overlap), 4),
            timings=TIMINGS, elapsed_s=round(time.perf_counter() - T0, 1))
        (OUT / 'hybrid_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
        print(json.dumps(manifest, ensure_ascii=False, indent=2))

## 11. Extra ZIPs: neighbour spans (EXPAND_NB) and bge document rerank (DOC_RERANK)

In [ ]:
# Copied from scripts/expand_chunks.py by scripts/build_hybrid_notebook.py. Edit the script and rebuild.
# Run in its own namespace EXPAND: its VARIANTS and one_match would shadow this notebook's.
import types

EXPAND_SOURCE = r'''"""Rebuild the chunk lists of the hybrid submission with neighbouring chunks merged into longer spans.

A reference passage counts as found when a submitted text of the same document covers >= 40% of it (token
LCS). Chunk recall (0.18) is far below document recall (0.47): the right documents are found, the passages
are not. Here the top-ranked chunks bring their neighbours (order - 1, order + 1) along, and contiguous
chunks of one text are merged into one span (the 50-word overlaps are dropped once). The byte budget per
query stays that of the hybrid submission, so lower-ranked chunks make room for the neighbours.

The document lists are kept as submitted; documents of new spans are appended (as in assemble()).
Optional per variant: n_docs extends the document list with the next pool documents (best chunk first);
quota reserves that share of the chunk bytes for ZH/EN texts; doc_quota swaps the lowest VI documents of the
list for the best ZH/EN pool documents (both lost on the leaderboard, 29/10); doc_rr re-orders the
candidate documents by weighted RRF of their submitted rank and document-reranker scores (doc_rank.parquet,
written by the doc-rerank notebook, kaggle_doc_rerank.ipynb), so the reranker decides which make the cut.
Runs on a Kaggle CPU session (Internet OFF is fine). Inputs:
  medxir-chunk-corpus      chunks/*/part-*.parquet, hash_groups.parquet
  medxir-hybrid-retrieval  pool.parquet, medxir-hybrid-submission.zip   (notebook output or uploaded dataset)
  doc_rank.parquet         only for doc_rr variants (in OUT_DIR or an input)
Output: one ZIP per variant in /kaggle/working (or OUT_DIR), plus expand_manifest.json.
"""
import glob
import json
import math
import os
import zipfile
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

# nb: number of top chunks whose neighbours are added (0 = only merge, no new neighbours);
#     'whole' = probe: full texts of the top documents instead of ranked chunks.
# n_docs: document list length (0 = as submitted). quota / doc_quota: min ZH/EN share of chunk bytes / documents.
# doc_rr: {score column of doc_rank.parquet: RRF weight}; w_base: RRF weight of the submitted document rank;
# keep_top: that many top candidates kept as submitted, the rest of the list re-ordered.
VARIANTS = {
    'nb0': dict(nb=0), 'nb10': dict(nb=10), 'nb30': dict(nb=30), 'nb60': dict(nb=60), 'wholedoc': dict(nb='whole'),
    'nb60-d300': dict(nb=60, n_docs=300),
    'nb60-q20': dict(nb=60, quota=0.2),
    'nb60-q20-dq20': dict(nb=60, quota=0.2, doc_quota=0.2),
    'nb60-drr-bge': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=1.0),
    'nb60-drr-bge-only': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=0.0),
    # Qwen3-Reranker-4B scores only the zone around the cut (ranks 120-360): the top 120 stay as submitted.
    'nb60-drr-qwen4b': dict(nb=60, doc_rr={'Qwen3-Reranker-4B': 1.0}, w_base=1.0, keep_top=120),
    'nb60-drr-qwen4b-only': dict(nb=60, doc_rr={'Qwen3-Reranker-4B': 1.0}, w_base=0.0, keep_top=120),
    'nb60-drr-both': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0, 'Qwen3-Reranker-4B': 1.0}, w_base=1.0,
                          keep_top=120),
}
ONLY = ['nb60-d300']   # variants to build this run; [] = all
BUDGET_SCALE = 1.08   # 87.5 MB hybrid ZIP * 1.08 stays below the 100 MB limit
MAX_ZIP_MB = 99.0
RRF_K = 60
INPUT = os.environ.get('INPUT_ROOT', '/kaggle/input')
OUT = Path(os.environ.get('OUT_DIR', '/kaggle/working'))


def one_match(pattern):
    hits = sorted(p for p in glob.glob(f'{INPUT}/**/{pattern}', recursive=True))
    if len(hits) != 1:
        for root in sorted(glob.glob(f'{INPUT}/*/*/*')) or sorted(glob.glob(f'{INPUT}/*')):
            print('input:', root, sorted(os.listdir(root))[:8] if os.path.isdir(root) else '')
    assert len(hits) == 1, f'{pattern}: expected one match, found {hits[:10]}'
    return Path(hits[0])


def load_submission():
    # An uploaded dataset is unzipped by Kaggle: accept the JSON too.
    hits = sorted(glob.glob(f'{INPUT}/**/medxir-hybrid-submission.json', recursive=True))
    if hits:
        return json.loads(Path(hits[0]).read_text(encoding='utf-8'))
    with zipfile.ZipFile(one_match('medxir-hybrid-submission.zip')) as archive:
        return json.loads(archive.read(archive.namelist()[0]))


def merge_run(rows):
    """One text out of chunks of one document text with consecutive orders (rows sorted by order)."""
    text, end = rows[0]['text'], rows[0]['end']
    for row in rows[1:]:
        text += row['text'][max(0, end - row['start']):]
        end = row['end']
    return text


def build_query(ranked, n_neighbour, budget, rows, by_pos, docs_of, quota=0.0):
    """Selected chunk ids for one query: ranked chunks (+ neighbours of the top ones) until the budget is used.

    quota > 0: a first pass caps VI chunks at (1 - quota) of the budget so ZH/EN chunks further down the pool
    get the rest; a second pass fills what ZH/EN could not use.
    """
    chosen, used, vi_used = [], 0, 0
    for vi_cap in ([budget * (1 - quota)] if quota else []) + [budget]:
        for rank, cid in enumerate(ranked):
            group = [cid]
            if rank < n_neighbour:
                sha, order = rows[cid]['sha'], rows[cid]['order']
                group += [by_pos[(sha, o)] for o in (order - 1, order + 1) if (sha, o) in by_pos]
            for c in group:
                if c in chosen:
                    continue
                cost = len(rows[c]['text'].encode('utf-8')) * len(docs_of(c))
                vi = rows[c].get('lang', 'vi') == 'vi'
                if used + cost > budget or (vi and vi_used + cost > vi_cap):
                    continue  # a long or wide-fanout chunk does not fit; smaller ones further down may
                chosen.append(c)
                used += cost
                vi_used += cost if vi else 0
    return chosen


def doc_list(docs, ranked, docs_of, lang_of, n_docs=0, doc_quota=0.0):
    """Submitted documents with the ZH/EN quota applied, then extended to n_docs with pool documents."""
    docs = list(docs)
    pool_docs = list(dict.fromkeys(d for cid in ranked for d in docs_of(cid)))
    if doc_quota:
        listed = set(docs)
        extra = [d for d in pool_docs if d not in listed and lang_of.get(d, 'vi') != 'vi']
        need = math.ceil(doc_quota * len(docs)) - sum(lang_of.get(d, 'vi') != 'vi' for d in docs)
        vi_slots = [i for i, d in enumerate(docs) if lang_of.get(d, 'vi') == 'vi']
        slots = vi_slots[len(vi_slots) - min(max(need, 0), len(extra), len(vi_slots)):]
        for i, doc in zip(slots, extra):   # lowest VI slots, best ZH/EN document first
            docs[i] = doc
    if n_docs > len(docs):
        listed = set(docs)
        docs += [d for d in pool_docs if d not in listed][:n_docs - len(docs)]
    return docs


def best_chunks(ranked, docs_of):
    """{doc: its pool chunks, best first}. Documents filled from beyond the reranked pool have none."""
    best = {}
    for cid in ranked:
        for doc in docs_of(cid):
            best.setdefault(doc, []).append(cid)
    return best


def doc_text(best, rows, by_text, lead_chars=400, n_chunks=1):
    """Text a document reranker sees: the start of the document text (title / lead) + its best pool chunks.

    best: the document's pool chunks, best first; the n_chunks best are shown in text order.
    """
    shown = sorted(best[:n_chunks], key=lambda c: rows[c]['order'])
    parts = [rows[c]['text'] for c in shown]
    lead = rows[by_text[rows[shown[0]]['sha']][0]]
    if lead['order'] != rows[shown[0]]['order']:
        parts.insert(0, lead['text'][:lead_chars])
    return '\n...\n'.join(parts)


def fuse_docs(candidates, scores, weights, w_base=1.0, k=RRF_K, offset=0):
    """Candidates (submitted order) re-ordered by weighted RRF of their position and reranker scores.

    scores: {column: {doc: score}}; a document without a score in a column gets no term from it.
    offset: submitted rank of the first candidate (when the documents above it are kept fixed).
    """
    fused = {doc: w_base / (k + offset + rank) for rank, doc in enumerate(candidates)}
    for column, weight in weights.items():
        scored = sorted((d for d in candidates if d in scores[column]), key=lambda d: -scores[column][d])
        for rank, doc in enumerate(scored):
            fused[doc] += weight / (k + rank)
    return sorted(candidates, key=lambda d: -fused[d])


def spans(chosen, rows):
    """Merge chosen chunks of the same text with consecutive orders. Returns [(first chunk id, text)] in rank order."""
    by_sha = {}
    for cid in chosen:
        by_sha.setdefault(rows[cid]['sha'], []).append(cid)
    first_rank = {cid: i for i, cid in enumerate(chosen)}
    out = []
    for cids in by_sha.values():
        cids = sorted(cids, key=lambda c: rows[c]['order'])
        run = [cids[0]]
        for c in cids[1:]:
            if rows[c]['order'] == rows[run[-1]]['order'] + 1:
                run.append(c)
            else:
                out.append(run)
                run = [c]
        out.append(run)
    out.sort(key=lambda run: min(first_rank[c] for c in run))
    return [(run[0], merge_run([rows[c] for c in run])) for run in out]


def build_whole_docs(ranked, docs, budget, rows, by_text, docs_of):
    """Probe: every chunk of the submitted documents, best document first, until the budget is used.

    Tells whether chunk recall is lost inside the right documents (recall jumps) or by missing documents.
    """
    text_of = {}
    for cid in ranked:
        for doc in docs_of(cid):
            text_of.setdefault(doc, rows[cid]['sha'])
    chosen, used = [], 0
    for doc in docs:
        if doc not in text_of:
            continue
        for c in by_text[text_of[doc]]:
            cost = len(rows[c]['text'].encode('utf-8')) * len(docs_of(c))
            if used + cost > budget:
                return chosen
            if c not in chosen:
                chosen.append(c)
                used += cost
    return chosen


def load_inputs(pool_path=None, base=None, chunk_files=None):
    """Pool ranking, base submission, every chunk of the candidate texts and the document groups.

    The hybrid notebook passes its own pool.parquet, in-memory submission and validated chunk parts;
    otherwise they are found under INPUT.
    """
    pool = pd.read_parquet(pool_path or one_match('pool.parquet'), columns=['query_id', 'chunk_id', 'final_rank'])
    pool = pool.sort_values(['query_id', 'final_rank'])
    ranked = {qid: g['chunk_id'].tolist() for qid, g in pool.groupby('query_id', sort=False)}
    base = load_submission() if base is None else base
    needed_sha_prefix = {cid.rsplit('-', 1)[0] for ids in ranked.values() for cid in ids}

    # Every chunk of the candidate texts: the neighbours are rows of the same text_sha256.
    rows, by_pos = {}, {}
    chunk_files = chunk_files or sorted(glob.glob(f'{INPUT}/**/chunks/*/part-*.parquet', recursive=True))
    assert chunk_files, 'no chunk parts: attach medxir-chunk-corpus'
    columns = ['chunk_id', 'text_sha256', 'rep_doc_id', 'order', 'start', 'end', 'chunk_text']
    for path in chunk_files:
        table = pq.read_table(path, columns=columns)
        prefix = pc.utf8_slice_codeunits(table['text_sha256'], 0, 16)
        table = table.filter(pc.is_in(prefix, value_set=pa.array(sorted(needed_sha_prefix))))
        lang = Path(path).parent.name   # chunks/<lang>/part-*.parquet
        for cid, sha, doc, order, start, end, text in zip(*(table[c].to_pylist() for c in columns)):
            rows[cid] = dict(sha=sha, doc=int(doc), order=int(order), start=int(start), end=int(end), text=text,
                             lang=lang)
            by_pos[(sha, int(order))] = cid
        print(Path(path).parent.name, Path(path).name, f'{len(rows):,} rows kept', flush=True)
    missing = {cid for ids in ranked.values() for cid in ids} - set(rows)
    assert not missing, f'{len(missing)} pool chunks not in the chunk corpus (chunk id != sha prefix?), e.g. {sorted(missing)[:3]}'

    groups = pd.read_parquet(one_match('hash_groups.parquet'), columns=['text_sha256', 'doc_ids', 'fanout_safe'])
    group = {sha: [int(d) for d in ids] for sha, ids, safe in
             zip(groups['text_sha256'], groups['doc_ids'], groups['fanout_safe']) if bool(safe)}
    del groups

    def docs_of(cid):
        return group.get(rows[cid]['sha']) or [rows[cid]['doc']]

    by_text = {}
    for cid in sorted(rows, key=lambda c: rows[c]['order']):
        by_text.setdefault(rows[cid]['sha'], []).append(cid)
    return dict(ranked=ranked, base=base, rows=rows, by_pos=by_pos, by_text=by_text, docs_of=docs_of,
                lang_of={doc: row['lang'] for cid, row in rows.items() for doc in docs_of(cid)})


def load_doc_scores():
    """{query_id: {column: {doc: score}}} from doc_rank.parquet (OUT_DIR first, then the inputs)."""
    path = OUT / 'doc_rank.parquet'
    frame = pd.read_parquet(path if path.exists() else one_match('doc_rank.parquet'))
    columns = [c for c in frame.columns if c not in ('query_id', 'doc_id', 'base_rank')]
    out = {}
    for qid, g in frame.groupby('query_id'):
        out[int(qid)] = {c: {int(d): float(s) for d, s in zip(g['doc_id'], g[c]) if s == s} for c in columns}
    return out


def build_variants(ctx, only=None, doc_scores=None):
    ranked, base, rows, by_pos, by_text, docs_of, lang_of = (
        ctx[k] for k in ('ranked', 'base', 'rows', 'by_pos', 'by_text', 'docs_of', 'lang_of'))
    budget = {e['id']: BUDGET_SCALE * sum(len(c['chunk_text'].encode('utf-8')) for c in e['relevant_chunks'])
              for e in base}
    only = ONLY if only is None else only
    manifest = {'budget_scale': BUDGET_SCALE, 'variants': {}}
    for name, spec in VARIANTS.items():
        if only and name not in only:
            continue
        if spec.get('doc_rr'):
            doc_scores = doc_scores or load_doc_scores()
            some = next(iter(doc_scores.values()))
            absent = [c for c in spec['doc_rr'] if c not in some]
            if absent:
                print(name, 'skipped: no doc_rank column', absent, flush=True)
                continue
        n_neighbour = spec['nb']
        scale = 1.0
        while True:
            submission, n_spans, n_chunks, n_docs, other_docs, other_bytes, all_bytes, moved = [], 0, 0, 0, 0, 0, 0, 0
            for entry in base:
                qid = entry['id']
                if n_neighbour == 'whole':
                    chosen = build_whole_docs(ranked[qid], entry['relevant_docs'], budget[qid] * scale,
                                              rows, by_text, docs_of)
                else:
                    chosen = build_query(ranked[qid], n_neighbour, budget[qid] * scale, rows, by_pos, docs_of,
                                         spec.get('quota', 0.0))
                if spec.get('doc_rr'):
                    q_scores = doc_scores.get(qid, {})
                    candidates = list(dict.fromkeys(
                        doc_list(entry['relevant_docs'], ranked[qid], docs_of, lang_of, n_docs=10 ** 6)
                        + [d for c in spec['doc_rr'] for d in q_scores.get(c, {})]))
                    keep = spec.get('n_docs') or len(entry['relevant_docs'])
                    fixed = candidates[:spec.get('keep_top', 0)]
                    docs = fixed + fuse_docs(candidates[len(fixed):], q_scores, spec['doc_rr'], spec.get('w_base', 1.0),
                                             offset=len(fixed))[:keep - len(fixed)]
                    moved += len(set(docs) - set(entry['relevant_docs']))
                else:
                    docs = doc_list(entry['relevant_docs'], ranked[qid], docs_of, lang_of,
                                    spec.get('n_docs', 0), spec.get('doc_quota', 0.0))
                for c in chosen:
                    size = len(rows[c]['text'].encode('utf-8'))
                    all_bytes += size
                    other_bytes += size if rows[c]['lang'] != 'vi' else 0
                listed = set(docs)
                chunks = []
                for cid, text in spans(chosen, rows):
                    for doc in docs_of(cid):
                        chunks.append(dict(doc_id=doc, chunk_text=text, chunk_order=rows[cid]['order']))
                        if doc not in listed:
                            listed.add(doc)
                            docs.append(doc)
                n_spans += len(chunks)
                n_docs += len(docs)
                other_docs += sum(lang_of.get(d, 'vi') != 'vi' for d in docs) / max(len(docs), 1)
                n_chunks += len(chosen)
                submission.append(dict(id=qid, relevant_docs=docs, relevant_chunks=chunks))
            zip_path = OUT / f'medxir-expand-{name}-submission.zip'
            json_name = zip_path.name.replace('.zip', '.json')
            with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=9) as archive:
                archive.writestr(json_name, json.dumps(submission, ensure_ascii=False, separators=(',', ':')))
            mb = zip_path.stat().st_size / 1e6
            if mb <= MAX_ZIP_MB:
                break
            scale *= 0.95 * MAX_ZIP_MB / mb
        stats = dict(spec, zip_mb=round(mb, 1), budget_scale_applied=round(scale, 3),
                     chunks_per_query=round(n_chunks / len(base), 1), entries_per_query=round(n_spans / len(base), 1),
                     docs_per_query=round(n_docs / len(base), 1),
                     zh_en_doc_share=round(other_docs / len(base), 3),
                     zh_en_chunk_byte_share=round(other_bytes / max(all_bytes, 1), 3))
        if spec.get('doc_rr'):
            stats['new_docs_per_query'] = round(moved / len(base), 1)   # documents not in the submitted list
        manifest['variants'][name] = stats
        print(name, stats, flush=True)
    (OUT / 'expand_manifest.json').write_text(json.dumps(manifest, indent=2))
    return manifest


def main():
    build_variants(load_inputs())
'''
EXPAND = types.ModuleType('expand_chunks')
exec(compile(EXPAND_SOURCE, 'expand_chunks.py', 'exec'), EXPAND.__dict__)

In [ ]:
if FAILED is None:
    with stage('expand_inputs'):
        # Every chunk of the pool texts (neighbours need start/end/order), read again from the validated parts.
        EXPAND_ON = not QUERY_LIMIT and (EXPAND_NB > 0 or DOC_RERANK != 'off')
        if not EXPAND_ON:
            print('no extra ZIPs (EXPAND_NB = 0 and DOC_RERANK off, or QUERY_LIMIT set)')
        else:
            started = time.perf_counter()
            EXPAND.OUT = OUT
            CTX = EXPAND.load_inputs(pool_path=OUT / 'pool.parquet', base=submission, chunk_files=CHUNK_FILES)
            TIMINGS['expand_inputs_s'] = round(time.perf_counter() - started, 1)
            note(f'expand inputs: {len(CTX["rows"]):,} chunk rows')

In [ ]:
if FAILED is None:
    with stage('doc_rerank'):
        # Candidate documents per query: the submitted N_DOCS, then the other pool documents (best chunk first), up to
        # DOC_RERANK_DEPTH. bge scores (query, document lead + best pool chunk); fan-out documents share a text: scored once.
        DOC_SCORES = None
        if EXPAND_ON and DOC_RERANK != 'off':
            from transformers import AutoModelForSequenceClassification, AutoTokenizer

            started = time.perf_counter()
            keys, cand_rows = {}, []   # (qi, text) -> documents sharing it
            for qi, entry in enumerate(submission):
                ranked = CTX['ranked'][entry['id']]
                best = EXPAND.best_chunks(ranked, CTX['docs_of'])
                cands = EXPAND.doc_list(entry['relevant_docs'], ranked, CTX['docs_of'], CTX['lang_of'], n_docs=10 ** 6)
                for rank, doc in enumerate(cands[:DOC_RERANK_DEPTH]):
                    cand_rows.append((entry['id'], doc, rank))
                    if doc in best:   # documents filled from beyond the pool have no chunk text: they keep their rank
                        text = EXPAND.doc_text(best[doc], CTX['rows'], CTX['by_text'], DOC_LEAD_CHARS)
                        keys.setdefault((qi, text), []).append(doc)
            pairs = list(keys)
            print('example document text:', pairs[0][1][:600].replace('\n', ' | '))

            tok = AutoTokenizer.from_pretrained(str(DOC_RERANK_PATH))
            model = AutoModelForSequenceClassification.from_pretrained(str(DOC_RERANK_PATH), torch_dtype=torch.bfloat16).cuda().eval()
            order = np.argsort([len(t) for _, t in pairs], kind='stable')   # similar lengths per batch: less padding
            values = np.empty(len(pairs), dtype=np.float32)
            note(f'document rerank: {len(cand_rows):,} candidates, {len(pairs):,} pairs')
            t0, next_report = time.perf_counter(), 0.05
            with torch.inference_mode():
                for start in range(0, len(order), 64):
                    index = order[start:start + 64]
                    enc = tok([QUERY_TEXT[pairs[i][0]] for i in index], [pairs[i][1] for i in index], padding=True,
                              truncation=True, max_length=512, return_tensors='pt').to('cuda')
                    values[index] = torch.sigmoid(model(**enc).logits.view(-1).float()).cpu().numpy()
                    done = (start + len(index)) / len(order)
                    if done >= next_report:
                        rate = (start + len(index)) / (time.perf_counter() - t0)
                        note(f'document rerank {done:.0%} | {rate:.0f} pairs/s | ETA {(len(order) - start) / rate / 60:.0f} min')
                        next_report += 0.05
            del model, tok
            torch.cuda.empty_cache()

            score = {(QUERY_IDS[qi], doc): float(v) for (qi, text), v in zip(pairs, values) for doc in keys[(qi, text)]}
            DOC_SCORES = {qid: {'bge-reranker-v2-m3': {}} for qid in QUERY_IDS}
            for (qid, doc), v in score.items():
                DOC_SCORES[qid]['bge-reranker-v2-m3'][doc] = v
            pd.DataFrame([(qid, doc, rank, score.get((qid, doc), np.nan)) for qid, doc, rank in cand_rows],
                         columns=['query_id', 'doc_id', 'base_rank', 'bge-reranker-v2-m3']) \
              .to_parquet(OUT / 'doc_rank.parquet', index=False, compression='zstd')
            TIMINGS['doc_rerank_s'] = round(time.perf_counter() - started, 1)

In [ ]:
if FAILED is None:
    with stage('expand'):
        # One ZIP per document ordering, all with the nb chunk spans; documents re-ordered only with DOC_RERANK.
        if EXPAND_ON:
            orderings = {'off': [None], 'fusion': [('drr-bge', 1.0)], 'only': [('drr-bge-only', 0.0)],
                         'both': [('drr-bge', 1.0), ('drr-bge-only', 0.0)]}[DOC_RERANK]
            names = []
            for ordering in orderings:
                name, spec = f'nb{EXPAND_NB}-d{N_DOCS}', dict(nb=EXPAND_NB)
                if ordering:
                    name += '-' + ordering[0]
                    spec.update(doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=ordering[1])
                EXPAND.VARIANTS[name] = spec
                names.append(name)
            expand_manifest = EXPAND.build_variants(CTX, only=names, doc_scores=DOC_SCORES)
            expand_manifest.update(n_docs=N_DOCS, doc_rerank=DOC_RERANK, doc_rerank_depth=DOC_RERANK_DEPTH,
                                   doc_lead_chars=DOC_LEAD_CHARS, timings=TIMINGS, elapsed_s=round(time.perf_counter() - T0, 1))
            (OUT / 'expand_manifest.json').write_text(json.dumps(expand_manifest, indent=2) + '\n')
            for name in names:
                note(f'medxir-expand-{name}-submission.zip: {expand_manifest["variants"][name]}')

## 12. Report

In [ ]:
note('finished; status ' + ('error' if FAILED else 'ok'))
if FAILED:
    (OUT / 'stage_status.json').write_text(json.dumps(
        dict(status='error', failed_stage=FAILED, error_file='retrieval_error.txt'), indent=2) + '\n')